In [2]:
"""
Stage 4 (ES vs LS) — Multi-classifier nested CV at five panel sizes
====================================================================

For each panel size in {5, 7, 10, 12, 15} genes (top-K by fold_inclusion
in consensus_panel.csv from the ES vs LS Stage-3 no-SMOTE run):

  Phase 0  Subset train/test to top-K consensus genes (saved as
           train_postfeature.csv / test_postfeature.csv inside the
           panel-specific output folder).

  Phase 1  Nested CV on the train half:
              outer = 9 stratified folds x 3 seeds (42, 123, 256) = 27 folds
              inner = 4 stratified folds, GridSearchCV with RobustScaler
                      INSIDE the pipeline (refit per inner fold).
           Inner-CV scoring metric: balanced_accuracy.
           Sample weighting: composite (sex x cell_type) stratum balance
           multiplied by class balance (LS vs ES inverse frequency),
           applied to RBF, Linear, GaussianNB, ElasticNet (the classifiers
           whose fit() supports sample_weight).

Class encoding: NEG=LS=0 (minority, ~29 train), POS=ES=1 (majority, ~42 train).

Inputs:
    train.csv      Outputs/stage2_split_stratum_aware_ESLS/train.csv
    test.csv       Outputs/stage2_split_stratum_aware_ESLS/test.csv
    metadata       Meta_data_ESLS.csv   (filtered to train / test sample IDs)
    consensus      Outputs/stage3_feature_selection_ESLS/consensus_panel.csv

Outputs per panel size (in ES vs LS/Outputs/stage4_classifiers_ESLS_topK/):
    train_postfeature.csv, test_postfeature.csv
    fold_results.csv               per-(seed,fold,classifier) HPs + metrics
    consensus_hyperparams.csv      mode HP per classifier across 27 folds
    final_metrics.csv              mean +- std for full metric battery
    roc_curves.png                 binary ROC per classifier
    pr_curves.png                  binary PR per classifier
    enet_fold_inclusion.csv        gene -> % of folds non-zero in ENet
    rfe_fold_inclusion.csv         gene -> % of folds in RFE support
"""

from __future__ import annotations

import json
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    cohen_kappa_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR  = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
ESLS_DIR      = PIPELINE_DIR / "ES vs LS"
SPLIT_DIR     = ESLS_DIR / "Outputs" / "stage2_split_stratum_aware_ESLS"
STAGE3_DIR    = ESLS_DIR / "Outputs" / "stage3_feature_selection_ESLS"
STAGE4_BASE   = ESLS_DIR / "Outputs"

CONSENSUS_PATH = STAGE3_DIR / "consensus_panel.csv"
TRAIN_PATH     = SPLIT_DIR / "train.csv"
TEST_PATH      = SPLIT_DIR / "test.csv"
META_PATH      = ESLS_DIR / "Meta_data_ESLS.csv"   # full metadata; filtered by sample IDs

CONDITION_COL = "condition"
PANEL_SIZES   = [5, 7, 10, 12, 15]
SEEDS         = [42, 123, 256]
N_OUTER       = 9
N_INNER       = 4
N_JOBS        = -1

# Class encoding: LS = 0 (negative, minority), ES = 1 (positive, majority)
NEG_LABEL = "LS"
POS_LABEL = "ES"


# ============================ Data helpers =================================
def load_meta(sample_ids) -> pd.DataFrame:
    """Read full Meta_data_ESLS.csv and align rows to sample_ids."""
    meta = pd.read_csv(META_PATH)
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    return meta.loc[sample_ids]


def to_binary_labels(y_str: np.ndarray) -> np.ndarray:
    """LS stays LS; everything else collapsed to ES (data is already binary)."""
    return np.where(np.asarray(y_str) == NEG_LABEL, NEG_LABEL, POS_LABEL)


def encode_binary(y_str_binary: np.ndarray) -> np.ndarray:
    """LS -> 0, ES -> 1."""
    return np.where(y_str_binary == POS_LABEL, 1, 0).astype(int)


def load_features_x_samples(path: Path, top_genes: List[str]) -> pd.DataFrame:
    """Load CSV; ensure genes are the row index; subset to top_genes (in order)."""
    df = pd.read_csv(path, index_col=0)
    if len(set(top_genes) & set(df.index.astype(str))) == 0:
        df = df.T
    present = [g for g in top_genes if g in df.index]
    missing = [g for g in top_genes if g not in df.index]
    if missing:
        print(f"  WARN: {len(missing)} genes missing in {path.name}")
    return df.loc[present]


def composite_sample_weights(meta: pd.DataFrame,
                             y_int: np.ndarray) -> np.ndarray:
    """Composite weight = confounder (sex x cell_type) stratum balance
    multiplied by class balance (LS vs ES inverse frequency), normalised
    to mean 1. The class term mirrors sklearn's class_weight='balanced':
    w_c = n / (n_classes * count_c)."""
    strata = meta["sex"].astype(str) + "_" + meta["cell_type"].astype(str)
    s_counts = strata.value_counts()
    n, k = len(strata), len(s_counts)
    w_conf = strata.map(lambda s: (n / k) / s_counts[s]).values.astype(float)

    classes, c_counts = np.unique(y_int, return_counts=True)
    n_cls = len(classes)
    class_w = {c: n / (n_cls * cnt) for c, cnt in zip(classes, c_counts)}
    w_class = np.array([class_w[y] for y in y_int], dtype=float)

    w = w_conf * w_class
    return w / w.mean()


# ============================ Classifiers ==================================
def get_classifiers(seed: int) -> Dict[str, Dict]:
    return {
        "mSVM-RBF": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="rbf", probability=True,
                                          random_state=seed))]),
            "grid": {"clf__C":     [0.1, 1, 10],
                     "clf__gamma": ["scale", 0.01, 0.1, 1.0]},
            "supports_sw": True,
        },
        "mSVM-Linear": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="linear", probability=True,
                                          random_state=seed))]),
            "grid": {"clf__C": [0.01, 0.1, 1, 10]},
            "supports_sw": True,
        },
        "mSVM-RFE": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("rfe", RFE(SVC(kernel="linear", probability=True,
                                              random_state=seed)))]),
            "grid": {"rfe__n_features_to_select": None,    # filled per panel
                     "rfe__estimator__C":         [0.1, 1, 10]},
            "supports_sw": False,
        },
        "kNN": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", KNeighborsClassifier())]),
            "grid": {"clf__n_neighbors": [3, 5, 7, 11, 15],
                     "clf__weights":     ["uniform", "distance"]},
            "supports_sw": False,
        },
        "GaussianNB": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", GaussianNB())]),
            "grid": {"clf__var_smoothing": np.logspace(-11, -7, 5)},
            "supports_sw": True,
        },
        "ElasticNet": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", LogisticRegression(
                                  penalty="elasticnet", solver="saga",
                                  max_iter=5000, tol=1e-3,
                                  random_state=seed))]),
            "grid": {"clf__C":        [0.01, 0.1, 1, 10],
                     "clf__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]},
            "supports_sw": True,
        },
    }


def rfe_grid_for_panel(k_top: int) -> List[int]:
    """Sensible n_features_to_select grid given the panel size."""
    base = sorted({max(2, k_top // 4), max(3, k_top // 2), k_top - 1, k_top})
    return [b for b in base if 2 <= b <= k_top]


# ============================ Metrics ======================================
def evaluate_binary(y_true: np.ndarray, y_pred: np.ndarray,
                    y_proba_pos: np.ndarray) -> Dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro"),
        "auroc":             roc_auc_score(y_true, y_proba_pos),
        "pr_auc":            average_precision_score(y_true, y_proba_pos),
        "sensitivity":       sens,
        "specificity":       spec,
        "mcc":               matthews_corrcoef(y_true, y_pred),
        "brier":             brier_score_loss(y_true, y_proba_pos),
        "cohen_kappa":       cohen_kappa_score(y_true, y_pred),
    }


def selected_genes_from_estimator(name: str, gs_result, gene_names: List[str]):
    best = gs_result.best_estimator_
    if name == "ElasticNet":
        coef = best.named_steps["clf"].coef_
        mask = (np.abs(coef) > 1e-10).any(axis=0)
        return [g for g, m in zip(gene_names, mask) if m]
    if name == "mSVM-RFE":
        rfe = best.named_steps["rfe"]
        return [g for g, m in zip(gene_names, rfe.support_) if m]
    return []


# ============================ Aggregation helpers ===========================
def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["balanced_accuracy", "macro_f1", "auroc", "pr_auc",
                   "sensitivity", "specificity", "mcc", "brier", "cohen_kappa"]
    rows = []
    for clf, sub in df.groupby("classifier"):
        row = {"classifier": clf, "n_folds": len(sub)}
        for m in metric_cols:
            row[f"{m}_mean"] = sub[m].mean()
            row[f"{m}_std"]  = sub[m].std()
        rows.append(row)
    return pd.DataFrame(rows)


def consensus_hyperparams(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for clf, sub in df.groupby("classifier"):
        params_list = [json.loads(p) for p in sub["best_params"]]
        keys = sorted({k for p in params_list for k in p.keys()})
        row = {"classifier": clf}
        for k in keys:
            vals = [p.get(k) for p in params_list]
            counter = Counter([str(v) for v in vals])
            mode_str, mode_count = counter.most_common(1)[0]
            row[k] = mode_str
            row[f"{k}__support_pct"] = round(100 * mode_count / len(vals), 1)
        rows.append(row)
    return pd.DataFrame(rows)


def fold_inclusion_table(sel_records, top_genes, n_folds_total):
    df = pd.DataFrame(sel_records)
    out = {}
    if df.empty:
        return out
    for clf in df["classifier"].unique():
        sub = df[df["classifier"] == clf]
        n_seen = sub.groupby("gene").size()
        full = pd.Series(0, index=top_genes, dtype=int)
        full.update(n_seen)
        out[clf] = pd.DataFrame({
            "gene": full.index,
            "n_folds_included": full.values,
            "fold_inclusion_pct":
                np.round(100 * full.values / n_folds_total, 1),
        }).sort_values("n_folds_included", ascending=False)
    return out


# ============================ Plot helpers =================================
def plot_roc_per_classifier(fold_df: pd.DataFrame, out_path: Path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for clf, sub in fold_df.groupby("classifier"):
        y_true_all = np.concatenate([np.asarray(t) for t in sub["y_true"]])
        y_proba_all = np.concatenate([np.asarray(p) for p in sub["y_proba_pos"]])
        fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
        auc = roc_auc_score(y_true_all, y_proba_all)
        ax.plot(fpr, tpr, label=f"{clf} (AUC={auc:.3f})", linewidth=1.7)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves across 27 outer folds (concatenated) — ES vs LS")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


def plot_pr_per_classifier(fold_df: pd.DataFrame, out_path: Path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for clf, sub in fold_df.groupby("classifier"):
        y_true_all = np.concatenate([np.asarray(t) for t in sub["y_true"]])
        y_proba_all = np.concatenate([np.asarray(p) for p in sub["y_proba_pos"]])
        prec, rec, _ = precision_recall_curve(y_true_all, y_proba_all)
        ap = average_precision_score(y_true_all, y_proba_all)
        ax.plot(rec, prec, label=f"{clf} (AP={ap:.3f})", linewidth=1.7)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("PR curves across 27 outer folds (concatenated) — ES vs LS")
    ax.legend(loc="lower left", fontsize=9)
    ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


# ============================ Per-classifier CV ============================
def run_classifier_cv(name, cfg, X, y_int, gene_names, meta,
                      fold_records, sel_records):
    print(f"\n--- {name} ---")
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=seed)
        for fold_i, (tr_idx, te_idx) in enumerate(skf.split(X, y_int)):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y_int[tr_idx], y_int[te_idx]
            sw_tr = composite_sample_weights(meta.iloc[tr_idx], y_int[tr_idx])

            inner = StratifiedKFold(n_splits=N_INNER, shuffle=True,
                                    random_state=seed * 1000 + fold_i)
            gs = GridSearchCV(
                estimator=cfg["pipe"],
                param_grid=cfg["grid"],
                scoring="balanced_accuracy",
                cv=inner, n_jobs=N_JOBS, refit=True,
            )

            fit_kwargs = {}
            if cfg["supports_sw"]:
                fit_kwargs["clf__sample_weight"] = sw_tr

            gs.fit(X_tr, y_tr, **fit_kwargs)

            y_pred  = gs.predict(X_te)
            y_proba = gs.predict_proba(X_te)
            y_proba_pos = y_proba[:, 1]
            metrics = evaluate_binary(y_te, y_pred, y_proba_pos)

            fold_records.append({
                "classifier":   name,
                "seed":         seed,
                "fold":         fold_i + 1,
                "best_params":  json.dumps(gs.best_params_),
                **metrics,
                "y_true":       y_te.tolist(),
                "y_proba_pos":  y_proba_pos.tolist(),
            })
            print(f"  seed={seed} fold={fold_i+1}  "
                  f"BA={metrics['balanced_accuracy']:.3f}  "
                  f"F1={metrics['macro_f1']:.3f}  "
                  f"AUROC={metrics['auroc']:.3f}  "
                  f"PR-AUC={metrics['pr_auc']:.3f}", flush=True)

            if name in ("ElasticNet", "mSVM-RFE"):
                sel = selected_genes_from_estimator(name, gs, gene_names)
                for g in sel:
                    sel_records.append({"classifier": name, "seed": seed,
                                        "fold": fold_i + 1, "gene": g})


# ============================ Per-panel run ================================
def run_for_panel(k_top: int):
    out_dir = STAGE4_BASE / f"stage4_classifiers_ESLS_top{k_top}"
    out_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 72)
    print(f"PANEL SIZE: top {k_top} consensus genes — output: {out_dir.name}")
    print("=" * 72)

    # ---- Top-K consensus genes ----
    consensus = pd.read_csv(CONSENSUS_PATH, index_col=0)
    if "fold_inclusion" not in consensus.columns:
        consensus.columns = ["fold_inclusion"]
    top_genes = (consensus.sort_values("fold_inclusion", ascending=False)
                          .head(k_top).index.tolist())
    print(f"Top genes loaded: {len(top_genes)}")

    train_df = load_features_x_samples(TRAIN_PATH, top_genes).round(4)
    test_df  = load_features_x_samples(TEST_PATH,  top_genes).round(4)
    train_df.to_csv(out_dir / "train_postfeature.csv")
    test_df.to_csv(out_dir / "test_postfeature.csv")
    print(f"Saved postfeature CSVs (train: {train_df.shape}, "
          f"test: {test_df.shape})")

    meta_train = load_meta(train_df.columns)
    y_train_str = meta_train[CONDITION_COL].values
    y_train_bin = to_binary_labels(y_train_str)
    y_train_int = encode_binary(y_train_bin)
    print(f"Class balance (ES=1, LS=0): "
          f"{dict(Counter(y_train_bin))}")

    X_train = train_df.T.values.astype(np.float64)
    gene_names = train_df.index.tolist()

    # ---- Build classifiers + RFE grid for this panel ----
    classifiers = get_classifiers(seed=42)
    classifiers["mSVM-RFE"]["grid"]["rfe__n_features_to_select"] = \
        rfe_grid_for_panel(k_top)
    print(f"RFE n_features_to_select grid: "
          f"{classifiers['mSVM-RFE']['grid']['rfe__n_features_to_select']}")

    # ---- Nested CV ----
    fold_records, sel_records = [], []
    for name, cfg in classifiers.items():
        run_classifier_cv(name, cfg, X_train, y_train_int, gene_names,
                          meta_train, fold_records, sel_records)

    fold_df = pd.DataFrame(fold_records)
    csv_cols = [c for c in fold_df.columns
                if c not in ("y_true", "y_proba_pos")]
    fold_df[csv_cols].to_csv(out_dir / "fold_results.csv", index=False)

    agg = aggregate_metrics(fold_df)
    agg.to_csv(out_dir / "final_metrics.csv", index=False)
    print("\nFinal metrics (mean across 27 folds):")
    print(agg.set_index("classifier")
          [["balanced_accuracy_mean", "macro_f1_mean",
            "auroc_mean", "pr_auc_mean"]].round(3).to_string())

    cons = consensus_hyperparams(fold_df)
    cons.to_csv(out_dir / "consensus_hyperparams.csv", index=False)
    print("\nConsensus hyperparameters (mode across 27 folds):")
    print(cons.set_index("classifier").to_string())

    plot_roc_per_classifier(fold_df, out_dir / "roc_curves.png")
    plot_pr_per_classifier(fold_df,  out_dir / "pr_curves.png")

    n_folds_total = N_OUTER * len(SEEDS)
    inc_tables = fold_inclusion_table(sel_records, top_genes, n_folds_total)
    if "ElasticNet" in inc_tables:
        inc_tables["ElasticNet"].to_csv(
            out_dir / "enet_fold_inclusion.csv", index=False)
    if "mSVM-RFE" in inc_tables:
        inc_tables["mSVM-RFE"].to_csv(
            out_dir / "rfe_fold_inclusion.csv", index=False)

    print(f"\nOutputs in: {out_dir}")


def main():
    print("=" * 72)
    print("STAGE 4 (ES vs LS) — Multi-classifier nested CV")
    print(f"Panel sizes: {PANEL_SIZES}")
    print(f"Outer={N_OUTER} x seeds={SEEDS} = {N_OUTER * len(SEEDS)} folds; "
          f"inner={N_INNER}")
    print("=" * 72)
    for k in PANEL_SIZES:
        run_for_panel(k)


if __name__ == "__main__":
    main()


STAGE 4 (ES vs LS) — Multi-classifier nested CV
Panel sizes: [5, 7, 10, 12, 15]
Outer=9 x seeds=[42, 123, 256] = 27 folds; inner=4

PANEL SIZE: top 5 consensus genes — output: stage4_classifiers_ESLS_top5
Top genes loaded: 5
Saved postfeature CSVs (train: (5, 71), test: (5, 31))
Class balance (ES=1, LS=0): {np.str_('LS'): 30, np.str_('ES'): 41}
RFE n_features_to_select grid: [2, 3, 4, 5]

--- mSVM-RBF ---
  seed=42 fold=1  BA=0.500  F1=0.467  AUROC=0.688  PR-AUC=0.733
  seed=42 fold=2  BA=0.750  F1=0.733  AUROC=0.938  PR-AUC=0.950
  seed=42 fold=3  BA=0.875  F1=0.873  AUROC=0.812  PR-AUC=0.893
  seed=42 fold=4  BA=0.700  F1=0.619  AUROC=0.933  PR-AUC=0.967
  seed=42 fold=5  BA=0.900  F1=0.873  AUROC=0.933  PR-AUC=0.967
  seed=42 fold=6  BA=0.733  F1=0.733  AUROC=0.867  PR-AUC=0.943
  seed=42 fold=7  BA=0.800  F1=0.750  AUROC=0.733  PR-AUC=0.885
  seed=42 fold=8  BA=0.700  F1=0.619  AUROC=0.867  PR-AUC=0.927
  seed=42 fold=9  BA=0.750  F1=0.708  AUROC=0.833  PR-AUC=0.887
  seed=123 fold